# **1. Seq2Seq**
시퀀스투시퀀스(Sequence-to-Sequence, Seq2Seq) 모델은 입력 시퀀스를 받아 출력 시퀀스를 생성하는 신경망 아키텍처로, 
<br>주로 자연어 처리 분야에서 번역, 요약, 질의응답 등에 활용됩니다. 
<br>Seq2Seq 모델은 일반적으로 인코더(Encoder)와 디코더(Decoder) 두 부분으로 구성되며, 
<br>인코더는 입력 문장을 고정된 크기의 컨텍스트 벡터로 변환하고, 디코더는 이를 기반으로 출력 문장을 생성합니다. 
<br>RNN, LSTM, GRU 등의 순환 신경망을 활용하며, 
<br>최근에는 어텐션 메커니즘과 트랜스포머 모델이 도입되어 더 긴 문맥을 효과적으로 처리할 수 있게 되었습니다.
<br>[[논문](https://arxiv.org/abs/1409.3215)]
https://arxiv.org/pdf/1409.3215

이건 트랜스포머
https://arxiv.org/abs/1706.03762
<br>꼭 봤으면 하는 논문 2개

- Seq2Seq는 입력 시퀀스를 받아 다른 출력 시퀀스로 바꾸는 구조
- 대표적인 예는 기계 번역
    - I love coffee > 나는 커피를 좋아해
    - I love coffee > Encoder > 문장 정보 압축 > Decoder > \<BOS\> 나는 / 커피를 / 좋아해 \<EOS\> (EOS는 끝났다는 걸 의미)
    - \<BOS\>: 문장 생성을 시작한다는 특별 토큰
    - \<EOS\>: 문장이 끝났다는 특별 토큰
    - Decoder는 이전에 생성한 토큰을 이용해 다음 토큰을 예측
- 고전적인 Seq2Seq는 보통 Encoder + Decoder로 구성되며, RNN/LSTM/GRU 등을 사용
    - Encoder: 입력을 순서대로 읽으며 hidden state에 정보를 누적
    - Decoder: Encoder가 만든 정보를 바탕으로 출력 토큰을 한 개씩 생성. 이전에 생성한 토큰을 이용해 다음 토큰을 예측. Autogressive(자기회귀)

![Seq2Seq](./images/Seq2Seq.png)

### Seq2Seq의 문제점
- 초기 Seq2Seq는 Encoder의 마지막 hidden state 하나를 Decoder에 전달하는 방식이 많이 사용되었음
- 짧은 문장이라면 괜찮지만 문장이 길어지면 마지막 hidden state 하나가 앞부분의 세부 정보까지 모두 기억해야 함. 이처럼 많은 정보를 제한된 하나의 표현에 압축하면서 중요한 정보가 손실될 수 있음(정보 병목, Bottleneck)

- RNN은 t 시점 계산 > t + 1 시점 게산처럼 앞 계산이 끝나야 다음 계산을 할 수 있으므로 긴 문장에서 학습이 느리고 병렬 처리에도 불리

 # **2. Attention 메커니즘**
- Attention은 현재 토큰이 다른 토큰들을 얼마나 참고할지 점수로 계산하는 방법
- 점수를 Softmax로 바꾸면 각 토큰을 섞는 Attention Weight가 됨
- Transformer의 Self-Attention에서는 문장 안의 모든 토큰이 서로의 관게를 직접 계산할 수 있음

```
예를들어 "나는 오늘 학교에서 수학 시험을 봤다" 에서 어떤 토큰을 해석할 때 항상 시험만 중요한 것은 아님. 
어떤 토큰을 처리하느냐와 학습된 관계에 따라 중요한 대상이 달라짐.
```

### 1. 토큰을 숫자 벡터로 바꾸기
- 신경망은 "커피 한잔 어때" 같은 문자열을 그대로 계산할 수 없음
- 문장 > Token > Token ID > Emnedding Layer > Embedding Vector
    - 커피: 0
    - 한잔: 1
    - 어때: 2
    - 0, 1, 2 자체는 의미가 없음. 단엉를 찾기 위한 번호일 뿐.
    - nn.Embedding이 번호를 학습 가능한 실수 벡터로 바꿈

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(2026)

token = ['커피', '한잔', '어때']
token_ids = torch.tensor([0, 1, 2])
print('토큰: ', token)
print('토큰 ID: ', token_ids)
print('토큰 ID shape: ', token_ids.shape)

토큰:  ['커피', '한잔', '어때']
토큰 ID:  tensor([0, 1, 2])
토큰 ID shape:  torch.Size([3])


### 2. nn.Embedding
- nn.Emnedding(num_embedding = 3, embedding_dim = 4)는 3행 * 4열의 학습 가능한 표

```
TokenI ID 0(커피) > [실수, 실수, 실수, 실수]
TokenI ID 1(한잔) > [실수, 실수, 실수, 실수]
TokenI ID 2(어때) > [실수, 실수, 실수, 실수]
```
- num_embeddings = 3: 사용할 수 있는 Token ID의 개수
- embedding_dim = 4: 토큰 하나를 표현하는 숫자의 개수
- embedding.weight.shape == (3, 4)

In [2]:
embedding = nn.Embedding(num_embeddings=3, embedding_dim=4)
print(embedding)
print(embedding.weight)

Embedding(3, 4)
Parameter containing:
tensor([[ 0.3655,  1.9065,  0.2169,  0.2963],
        [-1.0160, -0.6064,  0.1744, -0.6306],
        [-0.0954,  0.3191, -1.4710,  0.2610]], requires_grad=True)


In [3]:
X = embedding(token_ids)
print("임베딩 결과 X: ", X)
print("shape: ", X.shape)
print('토큰 수: ', X.shape[0])
print('임베딩 차원: ', X.shape[1])

임베딩 결과 X:  tensor([[ 0.3655,  1.9065,  0.2169,  0.2963],
        [-1.0160, -0.6064,  0.1744, -0.6306],
        [-0.0954,  0.3191, -1.4710,  0.2610]], grad_fn=<EmbeddingBackward0>)
shape:  torch.Size([3, 4])
토큰 수:  3
임베딩 차원:  4


### 3. Self-Attention
같은 시퀀스 안의 토큰들이 서로를 참고하는 Attention
```
"커피 한잔 어때"
커피 > 커피, 한잔, 어때
한잔 > 커피, 한잔, 어때
어때 > 커피, 한잔, 어때
```

> 토큰이 3개이므로 관계 점수표는 3*3

> Transformer가 임베딩 그대로 복사해서 비교하는 것이 아니라, 서로 다른 학습 가능한 선형 변환을 거쳐 만듦

### 4. Query, Key, Value(W, K, V)
- Query(Q): "나는 지금 어떤 정보를 찾고 있지?". XW_Q
- Key(K): "나는 어떤 특징을 가진 정보이지?". XW_K
- Value(V): "선택되면 실제로 전달할 내용은 이것이야". XW_V


In [4]:
embed_dim = 4 # 표현할 수 있는 개수
W_Q = nn.Linear(embed_dim, embed_dim, bias=False)
W_K = nn.Linear(embed_dim, embed_dim, bias=False)
W_V = nn.Linear(embed_dim, embed_dim, bias=False)

Q = W_Q(X)
K = W_K(X)
V = W_V(X)

print("X: ", X.shape)
print("Q: ", Q.shape)
print("K: ", K.shape)
print("V: ", V.shape)
print("Q = ", Q)
print("K = ", K)
print("V = ", V)

X:  torch.Size([3, 4])
Q:  torch.Size([3, 4])
K:  torch.Size([3, 4])
V:  torch.Size([3, 4])
Q =  tensor([[-0.3668, -0.8192,  0.1926, -0.0615],
        [ 0.0741,  0.5607, -0.4887, -0.2062],
        [-0.7271, -0.7872,  0.5736,  0.0966]], grad_fn=<MmBackward0>)
K =  tensor([[ 0.2404, -0.5815,  0.5650, -0.2048],
        [-0.4260,  0.1304, -0.2097,  0.2134],
        [ 0.6014, -0.1235,  0.1331,  0.7654]], grad_fn=<MmBackward0>)
V =  tensor([[ 0.1731,  0.7597, -0.1451,  0.0721],
        [-0.0511,  0.2425,  0.5725, -0.4278],
        [-0.2781, -0.3329, -0.1504,  0.5977]], grad_fn=<MmBackward0>)


### 5. Q와 K로 관계 점수 계산
현재 토큰의 Query와 모든 토큰의 Key를 내적하여 관련도 점수를 계산


```
현재
Q(3, 4), K(3, 4)이라 4x3, 3x4 계산 못함 
그래서 바꿔줌
Q(3, 4), K.T(4, 3) 계산 가능
```
-----------------------------------------------

```
scores = Q @ K.T
```
- Q: (3, 4)
- K.T: (4, 3)
- 결과: (3, 3)

커피 한잔 어때

> 결과의 행(row)은 질문하는 Query 토큰, 열(column)은 참고 대상 Key 토큰
> 예) scores[0, 2]는 커피의 Query와 어때의 Key를 비교한 원시 점수
